# Regression Sensitivity

## Calibrating omitted-variable sensitivity with partial R-squared

Consider a linear regression in which $Y$ is the outcome, $D$ is the treatment or exposure of interest, and $X$ contains the included controls. The reduced equation is

$$
Y=\tau D+X\beta+\varepsilon.
$$

The coefficient $\tau$ is the reported association between $D$ and $Y$ after controlling for $X$. The vector $\beta$ contains the coefficients on the included controls, and $\varepsilon$ is the remaining regression error. If an omitted variable $Z$ is added, the full equation becomes

$$
Y=\tau_ZD+X\beta_Z+\gamma Z+\varepsilon_Z.
$$

Here, $\tau_Z$ is the exposure coefficient after adjustment for $Z$, while $\gamma$ measures the conditional relation between $Z$ and $Y$. Sensitivity analysis describes how $\tau_Z$ changes as the conditional strength of $Z$ is varied. It does not prove that a particular omitted variable exists.

### Two Conditional Strength Measures

The first strength measure is the share of residual exposure variation explained by the omitted variable:

$$
R^2_{D\sim Z\mid X}=1-\frac{\operatorname{SSR}(D\mid X,Z)}{\operatorname{SSR}(D\mid X)}.
$$

The symbol $\operatorname{SSR}(D\mid X)$ is the residual sum of squares from regressing $D$ on the included controls. The numerator is the residual sum of squares after adding $Z$. The second strength measure is

$$
R^2_{Y\sim Z\mid D,X}=1-\frac{\operatorname{SSR}(Y\mid D,X,Z)}{\operatorname{SSR}(Y\mid D,X)}.
$$

This quantity is the additional share of residual outcome variation explained by $Z$ after $D$ and $X$ are already included. Both partial R-squared values lie between zero and one.

In [1]:
from pathlib import Path
import itertools
import sys
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sensemakr
warnings.filterwarnings('ignore', message='divide by zero encountered in scalar divide', category=RuntimeWarning, module='sensemakr\\.sensitivity_statistics')
import statsmodels.api as sm
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT
import RiskLabAI
from RiskLabAI.causal_factor_analysis import manski_ate_bounds
assert RiskLabAI.__version__ == '3.2.0'
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '05_partial_identification'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '05_partial_identification'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '05_partial_identification'
DATA_FILE = ensure_dataset(BOOK_ROOT, 'mpdta')
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})
VAR_LOOKUP = {'year': 'Year', 'countyreal': 'County identifier', 'lpop': 'Log population', 'lemp': 'Log employment', 'first_treat': 'First treatment year', 'treat': 'Treatment indicator'}

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return '0'
        if value.is_integer():
            return f'{int(value):,}'
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f'{value:.6g}'
        return f'{value:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    latex = latex.replace('\\begin{table}[htbp]', '\\begin{table}[htbp]\n\\centering', 1)
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)
    plt.close(figure)

def county_frame():
    raw = pd.read_csv(DATA_FILE)
    employment = raw.pivot(index='countyreal', columns='year', values='lemp')
    population = raw.pivot(index='countyreal', columns='year', values='lpop')
    first_treatment = raw.groupby('countyreal')['first_treat'].first()
    frame = pd.DataFrame({'County identifier': employment.index, 'Treatment group': (first_treatment.reindex(employment.index).to_numpy() > 0).astype(int), 'Log employment in 2003': employment[2003].to_numpy(), 'Log employment in 2006': employment[2006].to_numpy(), 'Log employment in 2007': employment[2007].to_numpy(), 'Log population in 2003': population[2003].to_numpy()})
    frame['Employment change, 2003 to 2007'] = frame['Log employment in 2007'] - frame['Log employment in 2003']
    frame['Employment increased, 2006 to 2007'] = (frame['Log employment in 2007'] > frame['Log employment in 2006']).astype(int)
    return (raw, frame)

def partial_r2(restricted, expanded):
    return float(1 - expanded.ssr / restricted.ssr)

def scalar(value):
    return float(np.asarray(value).item())
notation = pd.DataFrame([{'Symbol': 'Y', 'Description': 'Outcome'}, {'Symbol': 'D', 'Description': 'Treatment or exposure of interest'}, {'Symbol': 'X', 'Description': 'Included control variables'}, {'Symbol': 'Z', 'Description': 'Candidate omitted variable'}, {'Symbol': 'tau', 'Description': 'Exposure coefficient in the reduced regression'}, {'Symbol': 'tau_Z', 'Description': 'Exposure coefficient after adding Z'}, {'Symbol': 'R2 D with Z given X', 'Description': 'Partial R-squared of Z with the exposure'}, {'Symbol': 'R2 Y with Z given D and X', 'Description': 'Partial R-squared of Z with the outcome'}])
save_table(notation, 'sensitivity_notation', 'Notation for regression sensitivity analysis.', 'tab:cim5-sensitivity-notation', fit=True)
design = pd.DataFrame([{'Quantity': 'Observations', 'Value': 500}, {'Quantity': 'Random state', 'Value': 65}, {'Quantity': 'Coefficient from Z to D', 'Value': 0.8}, {'Quantity': 'Coefficient from X to D', 'Value': 0.3}, {'Quantity': 'Coefficient from D to Y', 'Value': 2}, {'Quantity': 'Coefficient from Z to Y', 'Value': 1.4}, {'Quantity': 'Coefficient from X to Y', 'Value': 1}])
save_table(design, 'sensitivity_generated_design', 'Declared parameters of the controlled omitted-variable calculation.', 'tab:cim5-sensitivity-generated-design')
rng = np.random.default_rng(65)
x, z, noise_d, noise_y = rng.normal(size=(4, 500))
d = 0.8 * z + 0.3 * x + noise_d
y = 2 * d + 1.4 * z + x + noise_y
reduced = sm.OLS(y, sm.add_constant(np.column_stack([d, x]))).fit()
full = sm.OLS(y, sm.add_constant(np.column_stack([d, x, z]))).fit()
d_reduced = sm.OLS(d, sm.add_constant(x)).fit()
d_full = sm.OLS(d, sm.add_constant(np.column_stack([x, z]))).fit()
r2d = partial_r2(d_reduced, d_full)
r2y = partial_r2(reduced, full)
adjusted = scalar(sensemakr.adjusted_estimate(r2d, r2y, estimate=float(reduced.params[1]), se=float(reduced.bse[1]), dof=int(reduced.df_resid)))
adjusted_se = scalar(sensemakr.adjusted_se(r2d, r2y, se=float(reduced.bse[1]), dof=int(reduced.df_resid)))
rv_generated = scalar(sensemakr.robustness_value(t_statistic=float(reduced.tvalues[1]), dof=int(reduced.df_resid)))
model_comparison = pd.DataFrame([{'Model': 'Reduced regression', 'Exposure estimate': reduced.params[1], 'Standard error': reduced.bse[1], 'Residual degrees of freedom': reduced.df_resid, 'Explained share': reduced.rsquared}, {'Model': 'Full regression', 'Exposure estimate': full.params[1], 'Standard error': full.bse[1], 'Residual degrees of freedom': full.df_resid, 'Explained share': full.rsquared}, {'Model': 'Sensitivity adjustment', 'Exposure estimate': adjusted, 'Standard error': adjusted_se, 'Residual degrees of freedom': full.df_resid, 'Explained share': np.nan}])
save_table(model_comparison, 'sensitivity_generated_models', 'Reduced, full, and sensitivity-adjusted exposure estimates.', 'tab:cim5-sensitivity-generated-models', fit=True)
strengths = pd.DataFrame([{'Strength measure': 'Partial R-squared with exposure', 'Value': r2d}, {'Strength measure': 'Partial R-squared with outcome', 'Value': r2y}, {'Strength measure': 'Robustness value for a zero estimate', 'Value': rv_generated}])
save_table(strengths, 'sensitivity_generated_strengths', 'Conditional strengths in the controlled calculation.', 'tab:cim5-sensitivity-generated-strengths', fit=True)
recovery = pd.DataFrame([{'Quantity': 'Adjusted estimate minus full estimate', 'Residual': adjusted - full.params[1], 'Pass': np.isclose(adjusted, full.params[1])}, {'Quantity': 'Adjusted standard error minus full standard error', 'Residual': adjusted_se - full.bse[1], 'Pass': np.isclose(adjusted_se, full.bse[1])}, {'Quantity': 'Zero-strength adjusted estimate minus supplied estimate', 'Residual': scalar(sensemakr.adjusted_estimate(0, 0, estimate=2, se=0.1, dof=30)) - 2, 'Pass': scalar(sensemakr.adjusted_estimate(0, 0, estimate=2, se=0.1, dof=30)) == 2}])
assert recovery['Pass'].all()
save_table(recovery, 'sensitivity_recovery_checks', 'Recovery and zero-strength checks.', 'tab:cim5-sensitivity-recovery', fit=True)
coefficient_data = model_comparison[['Model', 'Exposure estimate', 'Standard error']].copy()
coefficient_data.to_csv(FIGURE_DATA / 'sensitivity_generated_coefficients.csv', index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
axis.errorbar(coefficient_data['Exposure estimate'], coefficient_data['Model'], xerr=1.96 * coefficient_data['Standard error'], fmt='o', color=PALETTE['cyan'], ecolor=PALETTE['amber'], capsize=5)
axis.axvline(2, color=PALETTE['green'], linestyle='--', label='Declared coefficient')
axis.set_xlabel('Exposure coefficient')
axis.set_title('Omitting the Common Cause Changes the Estimate')
axis.legend()
finish_axis(axis)
save_figure(figure, 'sensitivity_generated_coefficients')
grid_values = np.linspace(0, 0.6, 31)
grid_records = []
for exposure_strength in grid_values:
    for outcome_strength in grid_values:
        value = scalar(sensemakr.adjusted_estimate(exposure_strength, outcome_strength, estimate=float(reduced.params[1]), se=float(reduced.bse[1]), dof=int(reduced.df_resid)))
        grid_records.append({'Partial R-squared with exposure': exposure_strength, 'Partial R-squared with outcome': outcome_strength, 'Adjusted estimate': value})
grid_frame = pd.DataFrame(grid_records)
grid_frame.to_csv(FIGURE_DATA / 'sensitivity_generated_surface.csv', index=False)
surface = grid_frame.pivot(index='Partial R-squared with outcome', columns='Partial R-squared with exposure', values='Adjusted estimate')
figure, axis = plt.subplots(figsize=(8.4, 6.2), constrained_layout=True)
image = axis.imshow(surface.to_numpy(), origin='lower', extent=[grid_values.min(), grid_values.max(), grid_values.min(), grid_values.max()], aspect='auto', cmap='viridis')
axis.scatter([r2d], [r2y], color=PALETTE['rose'], edgecolor=PALETTE['text'], s=70, label='Known omitted variable')
axis.set_xlabel('Partial R-squared with exposure')
axis.set_ylabel('Partial R-squared with outcome')
axis.set_title('Adjusted Estimate Across Omitted-Variable Strengths')
axis.legend()
figure.colorbar(image, ax=axis, label='Adjusted exposure estimate')
save_figure(figure, 'sensitivity_generated_surface')


,Symbol,Description
0,Y,Outcome
1,D,Treatment or exposure of interest
2,X,Included control variables
3,Z,Candidate omitted variable
4,tau,Exposure coefficient in the reduced regression
5,tau_Z,Exposure coefficient after adding Z
6,R2 D with Z given X,Partial R-squared of Z with the exposure
7,R2 Y with Z given D and X,Partial R-squared of Z with the outcome


,Quantity,Value
0,Observations,500
1,Random state,65
2,Coefficient from Z to D,0.8
3,Coefficient from X to D,0.3
4,Coefficient from D to Y,2
5,Coefficient from Z to Y,1.4
6,Coefficient from X to Y,1


,Model,Exposure estimate,Standard error,Residual degrees of freedom,Explained share
0,Reduced regression,2.7467,0.049046,497,0.885383
1,Full regression,2.067645,0.040553,496,0.952776
2,Sensitivity adjustment,2.067645,0.040553,496,


,Strength measure,Value
0,Partial R-squared with exposure,0.396123
1,Partial R-squared with outcome,0.587985
2,Robustness value for a zero estimate,0.877875


,Quantity,Residual,Pass
0,Adjusted estimate minus full estimate,-2.22045e-15,Pass
1,Adjusted standard error minus full standard error,-2.08167e-17,Pass
2,Zero-strength adjusted estimate minus supplied...,0,Pass


<Figure size 844.8x480 with 1 Axes>

<Figure size 806.4x595.2 with 2 Axes>

## Calibrating sensitivity to observed adjustment

The sensitivity calculation combines the two partial R-squared values with the reduced estimate and its standard error. A robustness value asks how strong an omitted variable would have to be, in both conditional relations, to move the estimate to a stated target. If the target is zero, the value summarizes the strength needed to remove the estimated sign under the sensitivity model. It is a compact description, not evidence that such a variable is absent.

In [2]:
identity_checks = pd.DataFrame([
    {"Case": "No exposure relation and no outcome relation", "Adjusted estimate": scalar(sensemakr.adjusted_estimate(0, 0, estimate=2, se=0.1, dof=30)), "Adjusted standard error": scalar(sensemakr.adjusted_se(0, 0, se=0.1, dof=30))},
    {"Case": "Moderate equal strengths", "Adjusted estimate": scalar(sensemakr.adjusted_estimate(0.1, 0.1, estimate=2, se=0.1, dof=30)), "Adjusted standard error": scalar(sensemakr.adjusted_se(0.1, 0.1, se=0.1, dof=30))},
    {"Case": "Stronger exposure relation", "Adjusted estimate": scalar(sensemakr.adjusted_estimate(0.3, 0.1, estimate=2, se=0.1, dof=30)), "Adjusted standard error": scalar(sensemakr.adjusted_se(0.3, 0.1, se=0.1, dof=30))},
])
save_table(identity_checks, "sensitivity_identity_cases", "Selected sensitivity calculations with supplied strengths.", "tab:cim5-sensitivity-identity-cases", fit=True)

diagonal_records = []
for strength in np.linspace(0, 0.6, 31):
    estimate = scalar(sensemakr.adjusted_estimate(strength, strength, estimate=float(reduced.params[1]), se=float(reduced.bse[1]), dof=int(reduced.df_resid)))
    standard_error = scalar(sensemakr.adjusted_se(strength, strength, se=float(reduced.bse[1]), dof=int(reduced.df_resid)))
    diagonal_records.append({"Equal partial R-squared": strength, "Adjusted estimate": estimate, "Lower 95 percent limit": estimate - 1.96 * standard_error, "Upper 95 percent limit": estimate + 1.96 * standard_error})
diagonal = pd.DataFrame(diagonal_records)
diagonal.to_csv(FIGURE_DATA / "sensitivity_equal_strength_path.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
axis.plot(diagonal["Equal partial R-squared"], diagonal["Adjusted estimate"], color=PALETTE["cyan"], linewidth=2.2, label="Adjusted estimate")
axis.fill_between(diagonal["Equal partial R-squared"], diagonal["Lower 95 percent limit"], diagonal["Upper 95 percent limit"], color=PALETTE["amber"], alpha=0.22, label="Regression interval")
axis.axhline(0, color=PALETTE["text"], linewidth=1)
axis.axvline(rv_generated, color=PALETTE["rose"], linestyle="--", label="Robustness value")
axis.set_xlabel("Equal partial R-squared values")
axis.set_ylabel("Exposure coefficient")
axis.set_title("Estimate Along an Equal-Strength Path")
axis.legend()
finish_axis(axis)
save_figure(figure, "sensitivity_equal_strength_path")

,Case,Adjusted estimate,Adjusted standard error
0,No exposure relation and no outcome relation,2,0.10171
1,Moderate equal strengths,1.942265,0.10171
2,Stronger exposure relation,1.886611,0.115328


<Figure size 844.8x480 with 1 Axes>

## Public County Regressions

The public calculation uses the 2003 to 2007 change in log employment as $Y$. The fixed treated-cohort indicator is $D$. Baseline log population and baseline log employment are observed controls. These regressions describe how the exposure coefficient changes when the two baseline variables are included. For each observed-variable benchmark, the adjustment direction follows the signs of the fitted relations: it can move the exposure estimate toward zero or away from zero. The calculation does not establish that either model identifies a causal effect.

In [3]:
raw, counties = county_frame()
y_public = counties['Employment change, 2003 to 2007'].to_numpy()
d_public = counties['Treatment group'].to_numpy()
population_public = counties['Log population in 2003'].to_numpy()
employment_public = counties['Log employment in 2003'].to_numpy()

def public_fit(columns):
    return sm.OLS(y_public, sm.add_constant(np.column_stack(columns))).fit()
public_models = [('Treatment indicator only', public_fit([d_public])), ('Add baseline population', public_fit([d_public, population_public])), ('Add baseline employment', public_fit([d_public, employment_public])), ('Add both baseline variables', public_fit([d_public, population_public, employment_public]))]
public_comparison = pd.DataFrame([{'Model': name, 'Exposure estimate': model.params[1], 'Standard error': model.bse[1], 'Test quantity': model.tvalues[1], 'Explained share': model.rsquared, 'Residual degrees of freedom': model.df_resid} for name, model in public_models])
save_table(public_comparison, 'mpdta_regression_models', 'Descriptive public county regressions.', 'tab:cim5-mpdta-regression-models', fit=True)
public_summary = pd.DataFrame([{'Quantity': 'County count', 'Value': len(counties)}, {'Quantity': 'Treatment fraction', 'Value': d_public.mean()}, {'Quantity': 'Mean employment change', 'Value': y_public.mean()}, {'Quantity': 'Employment-change standard deviation', 'Value': y_public.std(ddof=1)}])
save_table(public_summary, 'mpdta_regression_summary', 'Public county sample used in the descriptive sensitivity calculation.', 'tab:cim5-mpdta-regression-summary', fit=True)
full_public = public_models[-1][1]
benchmark_records = []
for omitted_name, omitted, retained in [('Baseline population', population_public, employment_public), ('Baseline employment', employment_public, population_public)]:
    outcome_restricted = public_fit([d_public, retained])
    exposure_restricted = sm.OLS(d_public, sm.add_constant(retained)).fit()
    exposure_expanded = sm.OLS(d_public, sm.add_constant(np.column_stack([retained, omitted]))).fit()
    exposure_strength = partial_r2(exposure_restricted, exposure_expanded)
    outcome_strength = partial_r2(outcome_restricted, full_public)
    reduce_direction = bool(abs(full_public.params[1]) <= abs(outcome_restricted.params[1]))
    adjusted_public = scalar(sensemakr.adjusted_estimate(exposure_strength, outcome_strength, estimate=float(outcome_restricted.params[1]), se=float(outcome_restricted.bse[1]), dof=int(outcome_restricted.df_resid), reduce=reduce_direction))
    adjusted_public_se = scalar(sensemakr.adjusted_se(exposure_strength, outcome_strength, se=float(outcome_restricted.bse[1]), dof=int(outcome_restricted.df_resid)))
    benchmark_records.append({'Omitted observed variable': omitted_name, 'Adjustment direction': 'Toward zero' if reduce_direction else 'Away from zero', 'Partial R-squared with exposure': exposure_strength, 'Partial R-squared with outcome': outcome_strength, 'Reduced estimate': outcome_restricted.params[1], 'Adjusted estimate': adjusted_public, 'Full estimate': full_public.params[1], 'Adjusted standard error': adjusted_public_se, 'Full standard error': full_public.bse[1]})
benchmarks = pd.DataFrame(benchmark_records)
assert np.allclose(benchmarks['Adjusted estimate'], benchmarks['Full estimate'])
assert np.allclose(benchmarks['Adjusted standard error'], benchmarks['Full standard error'])
save_table(benchmarks, 'mpdta_observed_covariate_benchmarks', 'Exact observed-covariate sensitivity benchmarks.', 'tab:cim5-mpdta-benchmarks', fit=True)
benchmark_book = benchmarks[['Omitted observed variable', 'Adjustment direction', 'Partial R-squared with exposure', 'Partial R-squared with outcome']].copy()
save_table(benchmark_book, 'mpdta_observed_covariate_benchmarks_book', 'Observed-covariate sensitivity benchmarks used in the chapter.', 'tab:cim5-mpdta-benchmarks-book', fit=True)
public_rv = scalar(sensemakr.robustness_value(t_statistic=float(full_public.tvalues[1]), dof=int(full_public.df_resid)))
robustness = pd.DataFrame([{'Calculation': 'Controlled generated regression', 'Robustness value': rv_generated, 'Exposure estimate': reduced.params[1], 'Standard error': reduced.bse[1]}, {'Calculation': 'Public regression with both baseline variables', 'Robustness value': public_rv, 'Exposure estimate': full_public.params[1], 'Standard error': full_public.bse[1]}])
save_table(robustness, 'sensitivity_robustness_values', 'Robustness values for the controlled and public calculations.', 'tab:cim5-sensitivity-robustness', fit=True)
public_plot = public_comparison[['Model', 'Exposure estimate', 'Standard error']].copy()
public_plot.to_csv(FIGURE_DATA / 'mpdta_regression_coefficients.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
axis.errorbar(public_plot['Exposure estimate'], public_plot['Model'], xerr=1.96 * public_plot['Standard error'], fmt='o', color=PALETTE['cyan'], ecolor=PALETTE['amber'], capsize=5)
axis.axvline(0, color=PALETTE['text'], linewidth=1)
axis.set_xlabel('Treatment-group coefficient')
axis.set_title('Public Coefficient Across Declared Control Sets')
finish_axis(axis)
save_figure(figure, 'mpdta_regression_coefficients')
benchmark_plot = benchmarks[['Omitted observed variable', 'Partial R-squared with exposure', 'Partial R-squared with outcome']].copy()
benchmark_plot.to_csv(FIGURE_DATA / 'mpdta_observed_covariate_benchmarks.csv', index=False)
figure, axis = plt.subplots(figsize=(8.4, 5.4), constrained_layout=True)
axis.scatter(benchmark_plot['Partial R-squared with exposure'], benchmark_plot['Partial R-squared with outcome'], s=85, color=[PALETTE['cyan'], PALETTE['amber']])
for _, row in benchmark_plot.iterrows():
    axis.annotate(row['Omitted observed variable'], (row['Partial R-squared with exposure'], row['Partial R-squared with outcome']), xytext=(7, 6), textcoords='offset points')
axis.set_xlabel('Partial R-squared with exposure')
axis.set_ylabel('Partial R-squared with outcome')
axis.set_title('Observed Baseline Variables as Sensitivity Benchmarks')
finish_axis(axis)
save_figure(figure, 'mpdta_observed_covariate_benchmarks')


,Model,Exposure estimate,Standard error,Test quantity,Explained share,Residual degrees of freedom
0,Treatment indicator only,-0.038538,0.022985,-1.676677,0.005613,498
1,Add baseline population,-0.044812,0.023097,-1.940186,0.014468,497
2,Add baseline employment,-0.040767,0.02317,-1.759481,0.006834,497
3,Add both baseline variables,-0.044163,0.022865,-1.931487,0.036174,496


,Quantity,Value
0,County count,500
1,Treatment fraction,0.382
2,Mean employment change,-0.008219
3,Employment-change standard deviation,0.250169


,Omitted observed variable,Adjustment direction,Partial R-squared with exposure,Partial R-squared with outcome,Reduced estimate,Adjusted estimate,Full estimate,Adjusted standard error,Full standard error
0,Baseline population,Away from zero,0.001461,0.029542,-0.040767,-0.044163,-0.044163,0.022865,0.022865
1,Baseline employment,Toward zero,7.20973e-05,0.022024,-0.044812,-0.044163,-0.044163,0.022865,0.022865


,Omitted observed variable,Adjustment direction,Partial R-squared with exposure,Partial R-squared with outcome
0,Baseline population,Away from zero,0.001461,0.029542
1,Baseline employment,Toward zero,7.20973e-05,0.022024


,Calculation,Robustness value,Exposure estimate,Standard error
0,Controlled generated regression,0.877875,2.7467,0.049046
1,Public regression with both baseline variables,0.083047,-0.044163,0.022865


<Figure size 864x499.2 with 1 Axes>

<Figure size 806.4x518.4 with 1 Axes>